# Federated Fraud Detection — Walkthrough

This notebook walks through the full pipeline step by step:
1. Load & explore the dataset
2. Non-IID partitioning across banks
3. Train a local baseline model
4. Run the federated learning loop
5. Compare FL vs centralised performance

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

from src.data_utils import (
    load_creditcard_dataset,
    load_synthetic_dataset,
    train_test_split_stratified,
    dirichlet_partition,
    compute_eda_stats,
    print_eda_report,
)
from src.federated_learning import FraudDetectorMLP, FederatedClient, FederatedServer

np.random.seed(42)

## Step 1 — Load Dataset

In [ ]:
# Use real data if available, otherwise synthetic
DATA_PATH = '../data/creditcard.csv'

if os.path.exists(DATA_PATH):
    X, y, feature_names = load_creditcard_dataset(DATA_PATH)
    print('Loaded Kaggle Credit Card Fraud dataset')
else:
    X, y, feature_names = load_synthetic_dataset(n_samples=8000)
    print('Kaggle CSV not found — using synthetic data')
    print('Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud')

X_train, X_test, y_train, y_test = train_test_split_stratified(X, y)

print(f'\nTotal samples : {len(y):,}')
print(f'Fraud rate    : {y.mean()*100:.3f}%')
print(f'Train / Test  : {len(y_train):,} / {len(y_test):,}')

## Step 2 — EDA

In [ ]:
stats = compute_eda_stats(X_train, y_train, feature_names)
print_eda_report(stats)

# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = [stats['class_counts'].get(0,0), stats['class_counts'].get(1,0)]
axes[0].bar(['Legitimate', 'Fraud'], counts, color=['#2563EB', '#DC2626'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + max(counts)*0.01, f'{v:,}', ha='center', fontweight='bold')

# Feature variance (top 10)
top_vars = X_train.var(axis=0)
top_idx = np.argsort(top_vars)[::-1][:10]
axes[1].barh([feature_names[i] for i in top_idx[::-1]], top_vars[top_idx[::-1]], color='#0F766E')
axes[1].set_title('Top 10 Features by Variance')
axes[1].set_xlabel('Variance')

plt.tight_layout()
plt.savefig('../results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Non-IID Partitioning

In [ ]:
NUM_BANKS = 6
ALPHA = 0.5  # lower = more non-IID

bank_data = dirichlet_partition(X_train, y_train, n_clients=NUM_BANKS, alpha=ALPHA)
bank_names = ['RetailBank-A','PrivateBank-B','CorporateBank-C',
              'TaxAuthority-D','CreditUnion-E','InvestBank-F']

print(f'Non-IID partitioning with Dirichlet alpha={ALPHA}\n')
print(f'{"Bank":<18} {"Samples":>8} {"Fraud":>8} {"Rate":>8}')
print('-' * 46)
for name, (bx, by) in zip(bank_names, bank_data):
    nf = (by==1).sum()
    print(f'{name:<18} {len(by):>8,} {nf:>8,} {by.mean()*100:>7.1f}%')

# Visualise distribution
fraud_rates = [(by==1).sum()/max(len(by),1)*100 for _,by in bank_data]
plt.figure(figsize=(9, 4))
bars = plt.bar(bank_names, fraud_rates, color='#1B3A5C', alpha=0.85)
plt.axhline(y_train.mean()*100, color='#DC2626', linestyle='--', label=f'Global avg ({y_train.mean()*100:.1f}%)')
plt.title('Fraud Rate per Bank (Non-IID Distribution)', fontsize=13, fontweight='bold')
plt.ylabel('Fraud Rate (%)')
plt.xticks(rotation=20, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('../results/noniid_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Local Baseline (Centralised)

In [ ]:
cw = (y_train==0).sum() / max((y_train==1).sum(), 1)

baseline = FraudDetectorMLP(
    input_dim=X_train.shape[1], hidden_dims=[64,32],
    learning_rate=0.005, class_weight=cw,
)
print('Training centralised baseline (all data, same architecture)...')
loss_hist = baseline.fit(X_train, y_train, epochs=15, batch_size=64)

preds = baseline.predict(X_test)
print('\nCentralised Baseline:')
print(classification_report(y_test, preds, target_names=['Legitimate','Fraud'], zero_division=0))

## Step 5 — Federated Learning Loop

In [ ]:
NUM_ROUNDS   = 10
LOCAL_EPOCHS = 15

clients = [
    FederatedClient(
        client_id=bank_names[i], X=bx, y=by,
        input_dim=X_train.shape[1], hidden_dims=[64,32],
        learning_rate=0.005, class_weight=cw,
    )
    for i, (bx,by) in enumerate(bank_data)
]

server = FederatedServer(
    input_dim=X_train.shape[1], hidden_dims=[64,32],
    learning_rate=0.005, class_weight=cw,
    clip_norm=2.0, dp_sigma=0.003, n_byzantine=1, fairness_alpha=0.3,
)

print(f'Running FL: {NUM_BANKS} banks × {NUM_ROUNDS} rounds × {LOCAL_EPOCHS} epochs\n')

for rnd in range(1, NUM_ROUNDS+1):
    global_weights = server.get_global_weights()
    c_weights, c_sizes, c_f1s, c_ids = [], [], [], []

    for client in clients:
        w, norm, f1 = client.train(global_weights, LOCAL_EPOCHS, 64, 2.0)
        c_weights.append(w); c_sizes.append(client.n_samples)
        c_f1s.append(f1);    c_ids.append(client.client_id)

    server.aggregate(c_weights, c_sizes, c_f1s, c_ids)
    m = server.evaluate(X_test, y_test, rnd)
    print(f'  Round {rnd:2d}  F1={m["f1"]:.4f}  AUC={m["auc"]:.4f}  Rec={m["recall"]:.4f}')

## Step 6 — Results & Comparison

In [ ]:
rounds = [m['round'] for m in server.history]
fl_f1  = [m['f1']    for m in server.history]
fl_auc = [m['auc']   for m in server.history]

from sklearn.metrics import f1_score, roc_auc_score
base_f1  = f1_score(y_test, baseline.predict(X_test), zero_division=0)
base_auc = roc_auc_score(y_test, baseline.predict_proba(X_test))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(rounds, fl_f1, 'o-', color='#2563EB', linewidth=2, markersize=6, label='FL Global Model')
axes[0].axhline(base_f1, color='#DC2626', linestyle='--', linewidth=1.5, label=f'Centralised ({base_f1:.3f})')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('F1 Score (Fraud)')
axes[0].set_title('F1 Score — FL vs Centralised', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1)

axes[1].plot(rounds, fl_auc, 's-', color='#0F766E', linewidth=2, markersize=6, label='FL Global Model')
axes[1].axhline(base_auc, color='#DC2626', linestyle='--', linewidth=1.5, label=f'Centralised ({base_auc:.3f})')
axes[1].set_xlabel('Round'); axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC — FL vs Centralised', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0.5, 1)

plt.tight_layout()
plt.savefig('../results/fl_vs_centralised.png', dpi=150, bbox_inches='tight')
plt.show()

final = server.history[-1]
print(f'\nFinal FL Model   — F1: {final["f1"]:.4f}  AUC: {final["auc"]:.4f}')
print(f'Centralised      — F1: {base_f1:.4f}  AUC: {base_auc:.4f}')
print(f'Privacy-accuracy gap: {abs(final["f1"] - base_f1):.4f} F1')